# RFECVSelector

SVM-RFE that chooses its own feature count. Inside an inner stratified
cross-validation it runs the elimination, scores every feature count it
passes, and keeps the count with the best mean score.

- **Cost:** `cv` times RFESelector, plus one final elimination on all the data.
  Inside the runner's own splits, this is a second, nested cross-validation.
- **Strength:** no `n_features` to guess.
- **Weakness:** the chosen count is itself noisy on small samples.
- **Classification only.** Wraps scikit-learn's `RFECV`.

In [1]:
import numpy as np
from sklearn.datasets import make_classification

from featsel.selectors import RFECVSelector

# 100 features, the first 5 informative.
X, y = make_classification(n_samples=300, n_features=100, n_informative=5,
                           n_redundant=0, shuffle=False, random_state=0)
truth = set(range(5))

## Letting cross-validation choose the count

In [2]:
selector = RFECVSelector(cv=5, random_state=0).fit(X, y)
chosen = selector.get_support(indices=True)
print('chosen count      :', selector.n_features_)
print('truly informative :', len(truth & set(chosen)), 'of 5')

chosen count      : 10
truly informative : 3 of 5


## The score behind the choice

In [3]:
results = selector.cv_results_
for n, score in zip(results['n_features'], results['mean_test_score']):
    marker = '  <- chosen' if n == selector.n_features_ else ''
    print(f'{n:4d} features  mean accuracy {score:.3f}{marker}')

   1 features  mean accuracy 0.760
  10 features  mean accuracy 0.777  <- chosen
  20 features  mean accuracy 0.767
  30 features  mean accuracy 0.770
  40 features  mean accuracy 0.777
  50 features  mean accuracy 0.723
  60 features  mean accuracy 0.710
  70 features  mean accuracy 0.730
  80 features  mean accuracy 0.733
  90 features  mean accuracy 0.733
 100 features  mean accuracy 0.723


## Where it fails

On a small sample the scores of neighbouring counts are within noise of each
other, so the chosen count jumps between subsamples of the same data.

In [4]:
rng = np.random.default_rng(0)
for draw in range(4):
    rows = rng.choice(len(y), size=60, replace=False)
    n = RFECVSelector(cv=5, random_state=0).fit(X[rows], y[rows]).n_features_
    print(f'subsample {draw}: chose {n} features')

subsample 0: chose 70 features


subsample 1: chose 50 features


subsample 2: chose 50 features


subsample 3: chose 40 features
